In [1]:
from wav2vec2decoder import Wav2Vec2Decoder, test
import plotly.express as px

/home/darinka/projects/ai-talent-hub-itmo-speech-course/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1

1. Implement all required decoding methods and add to the report comparison of their quality (either in normalized Levenshtein distance or [WER](https://en.wikipedia.org/wiki/Word_error_rate))

In [2]:
test_samples = [
    ("examples/sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
    ("examples/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
    ("examples/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
    ("examples/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
    ("examples/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
    ("examples/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
    ("examples/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
    ("examples/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
]

In [3]:
decoder = Wav2Vec2Decoder()
results = {audio_path: test(decoder, audio_path, target) for audio_path, target in test_samples}

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading the LM will be faster if you build a binary file.
Reading /home/darinka/projects/ai-talent-hub-itmo-speech-course/assignments/assignment2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


In [4]:
def plot_results(results):
    plot_data = []

    for audio_path, result in results.items():
        sample_num = audio_path.split('/')[-1].replace('.wav','')
        
        for method, data in result.items():
            plot_data.append({
                'Sample': sample_num,
                'Method': method,
                'Levenshtein Distance': data['levenshtein_distance']
            })

    fig = px.bar(
        plot_data,
        x='Sample',
        y='Levenshtein Distance', 
        color='Method',
        barmode='group',
        title='Levenshtein Distance by Decoding Method and Sample',
        labels={'Sample': 'Audio Sample'}
    )

    fig.show()

In [5]:
plot_results(results)

## Part 2

2. To see the effect of LM model, try loading larger N-gram LM model from [link](http://www.openslr.org/11/) and report how results are changed for the test audios

In [6]:
decoder = Wav2Vec2Decoder(lm_model_path="lm/4-gram.arpa.gz")
results_2 = {audio_path: test(decoder, audio_path, target) for audio_path, target in test_samples}
plot_results(results_2)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading the LM will be faster if you build a binary file.
Reading /home/darinka/projects/ai-talent-hub-itmo-speech-course/assignments/assignment2/lm/4-gram.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


In [7]:
results == results_2

True

## Part 3

3. Vary values of `beam_width`, `alpha` and `beta` parameters in all versions of beam search decoding and add your observations to the report

In [8]:
beam_widths = [10, 40, 70, 160, 130]
results_params = {}

for beam_width in beam_widths:
    decoder = Wav2Vec2Decoder(
        beam_width=beam_width,
    )
    
    param_name = f"beam={beam_width}"
    results_params[param_name] = {
        audio_path: test(decoder, audio_path, target) 
        for audio_path, target in test_samples
    }

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading the LM will be faster if you build a binary file.
Reading /home/darinka/projects/ai-talent-hub-itmo-speech-course/assignments/assignment2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading the LM will be faster if you build a binary file.
Reading 

In [9]:
plot_data = []
for param_set, results in results_params.items():
    for audio_path, data in results.items():
        sample_name = audio_path.split('/')[-1]
        for method, metrics in data.items():
            plot_data.append({
                'Sample': sample_name,
                'Parameters': param_set,
                'Method': method,
                'Levenshtein Distance': metrics['levenshtein_distance']
            })

fig = px.bar(
    plot_data,
    x='Sample',
    y='Levenshtein Distance',
    color='Parameters',
    facet_col='Method',
    barmode='group',
    title='Levenshtein Distance by Parameters, Method and Sample',
    labels={'Sample': 'Audio Sample'},
)

fig.update_layout(
    showlegend=True,
    legend_title_text='Parameter Sets',
    height=600
)

fig.show()

### alpha

In [10]:
alphas = [0.0, 0.5, 1.0, 1.5, 2.0]
results_alphas = {}

for alpha in alphas:
    decoder = Wav2Vec2Decoder(
        alpha=alpha,
    )
    
    param_name = f"alpha={alpha}"
    results_alphas[param_name] = {
        audio_path: test(decoder, audio_path, target) 
        for audio_path, target in test_samples
    }


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading the LM will be faster if you build a binary file.
Reading /home/darinka/projects/ai-talent-hub-itmo-speech-course/assignments/assignment2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading the LM will be faster if you build a binary file.
Reading 

In [11]:
plot_data = []
for param_set, results in results_alphas.items():
    for audio_path, data in results.items():
        sample_name = audio_path.split('/')[-1]
        for method, metrics in data.items():
            plot_data.append({
                'Sample': sample_name,
                'Parameters': param_set,
                'Method': method,
                'Levenshtein Distance': metrics['levenshtein_distance']
            })

fig = px.bar(
    plot_data,
    x='Sample',
    y='Levenshtein Distance',
    color='Parameters',
    facet_col='Method',
    barmode='group',
    title='Levenshtein Distance by Parameters, Method and Sample',
    labels={'Sample': 'Audio Sample'},
)

fig.update_layout(
    showlegend=True,
    legend_title_text='Parameter Sets',
    height=600
)

fig.show()

### beta

In [12]:
betas = [0.0, 0.5, 1.0, 1.5, 2.0]
results_betas = {}

for beta in betas:
    decoder = Wav2Vec2Decoder(
        beta=beta,
    )
    
    param_name = f"beta={beta}"
    results_betas[param_name] = {
        audio_path: test(decoder, audio_path, target) 
        for audio_path, target in test_samples
    }



Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading the LM will be faster if you build a binary file.
Reading /home/darinka/projects/ai-talent-hub-itmo-speech-course/assignments/assignment2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loading the LM will be faster if you build a binary file.
Reading 

In [13]:
plot_data = []
for param_set, results in results_betas.items():
    for audio_path, data in results.items():
        sample_name = audio_path.split('/')[-1]
        for method, metrics in data.items():
            plot_data.append({
                'Sample': sample_name,
                'Parameters': param_set,
                'Method': method,
                'Levenshtein Distance': metrics['levenshtein_distance']
            })

fig = px.bar(
    plot_data,
    x='Sample',
    y='Levenshtein Distance',
    color='Parameters',
    facet_col='Method',
    barmode='group',
    title='Levenshtein Distance by Parameters, Method and Sample',
    labels={'Sample': 'Audio Sample'},
)

fig.update_layout(
    showlegend=True,
    legend_title_text='Parameter Sets',
    height=600
)

fig.show()